# Transformer Models

This notebook covers the full transformer stack  -  from tokenizers and attention primitives up through complete task models.

## Class hierarchy and integration map

```
Tokenizers                       -> produce (B, T, embedding_dim)
  ContinuousInputTokenizer         used inside: TokenToTokenTransformer, TokenToVectorTransformer,
  DiscreteTokenTokenizer                        TokenToClassTransformer, ConditionedTokenTransformer
  SetTokenizer                     (alias of ContinuousInputTokenizer)
  PatchTokenizerND                 used inside: PatchTransformerND

Attention
  MultiHeadSelfAttention           used inside: TransformerBlock
  MultiHeadCrossAttention          used inside: TransformerBlock (when use_cross_attention=True)

FeedForward                        used inside: TransformerBlock
TransformerBlock                   used inside: TransformerStack
TransformerStack                   used inside: all full transformer models

Pooling & Heads
  TokenPooling                     used inside: PooledHead, TokenToVectorTransformer,
                                                TokenToClassTransformer
  TokenwiseHead / TokenDecoder     used inside: QuerySetDecoder
  PooledHead
    ClassificationHead             (alias)
    RegressionHead                 (alias)

Conditioning
  SinusoidalTimeEmbedding          used inside: TimeEmbeddingMLP
  TimeEmbeddingMLP                 used inside: TransformerConditioningBuilder
  TransformerConditioningBuilder   used inside: ConditionedTokenTransformer
  ConditionTokenProjector          standalone prepend-token utility

Decoders
  PatchDecoderND                   pairs with PatchTokenizerND / PatchTransformerND
  QuerySetDecoder                  standalone structured output decoder

Full models
  TokenToTokenTransformer          seq -> seq
  TokenToVectorTransformer         seq -> vector
  TokenToClassTransformer          seq -> logits
  ConditionedTokenTransformer      seq -> seq  (conditioned)
  PatchTransformerND               spatial -> grid/tokens/vector
```

## Task selection guide

| Task | Recommended class |
|---|---|
| Continuous feature sequence (time series, point cloud) -> tokens | `ContinuousInputTokenizer` |
| Discrete token IDs (text, codebook) -> embeddings | `DiscreteTokenTokenizer` |
| Images / volumes -> patch tokens | `PatchTokenizerND` |
| Sequence classification (sentiment, activity) | `TokenToClassTransformer` |
| Sequence embedding / retrieval | `TokenToVectorTransformer` |
| Sequence-to-sequence (denoising, transcription) | `TokenToTokenTransformer` |
| Vision Transformer (image -> grid/vector/tokens) | `PatchTransformerND` |
| Diffusion / conditional generation | `ConditionedTokenTransformer` |
| Custom transformer architecture | assemble `TransformerStack` + heads manually |

In [ ]:
import torch

from ml_suite.models.transformer import (
    ContinuousInputTokenizer, DiscreteTokenTokenizer, SetTokenizer, PatchTokenizerND,
    MultiHeadSelfAttention, MultiHeadCrossAttention,
    FeedForward, TransformerBlock, TransformerStack,
    TokenPooling,
    TokenwiseHead, ClassificationHead, RegressionHead,
    SinusoidalTimeEmbedding, TimeEmbeddingMLP,
    TransformerConditioningBuilder, ConditionTokenProjector, PatchDecoderND, QuerySetDecoder,
    TokenToTokenTransformer, TokenToVectorTransformer, TokenToClassTransformer,
    ConditionedTokenTransformer, PatchTransformerND,
)

---
## 1. Tokenizers

Tokenizers convert raw inputs into token tensors `(B, T, embedding_dim)`  -  the universal input format for all transformer models in this library.

> **Note on naming:** "Tokenizers" here means the components that convert raw modality inputs into the `(B, T, embedding_dim)` format the transformer expects — including input projection for continuous features, embedding lookup for discrete IDs, and patch extraction for spatial data. Upstream segmentation (e.g. BPE) is handled outside these classes and is out of scope.

**How to pick:**
- Continuous features per timestep -> `ContinuousInputTokenizer`
- Unordered set of elements (no sequence order) -> `SetTokenizer` (alias of `ContinuousInputTokenizer`)
- Integer token IDs (text, codebook) -> `DiscreteTokenTokenizer`
- Spatial grid (images, volumes, 1D signals) -> `PatchTokenizerND`

### 1.1 `ContinuousInputTokenizer`

Projects a sequence of continuous feature vectors into embedding space via a small MLP.

**Integrates into:** used as the input projection inside `TokenToTokenTransformer`, `TokenToVectorTransformer`, `TokenToClassTransformer`, and `ConditionedTokenTransformer`  -  all these models use it under the hood to lift `input_dim` to `embedding_dim`.

**Use standalone when:** you are assembling a custom transformer from `TransformerStack` and need to pre-project your input.

**Use for:** STFT frames, IMU sequences, point clouds, tabular sequences, any per-timestep continuous features.

In [2]:
tokenizer = ContinuousInputTokenizer(
    input_dim=16, embedding_dim=128, num_layers=2, activation="silu",
)

# (batch, tokens, raw_features)
# Real: STFT frames -> (B, n_frames, n_fft_bins)
#       IMU sequence -> (B, 500, 6)  (500 timesteps, 6 axes)
#       point cloud  -> (B, 1024, 3)
x      = torch.randn(2, 32, 16)
tokens = tokenizer(x)
print(f"{x.shape} -> {tokens.shape}")

torch.Size([2, 32, 16]) -> torch.Size([2, 32, 128])


### 1.2 `SetTokenizer`

Identical to `ContinuousInputTokenizer`  -  the different name signals to readers that the input has no natural order.
Use this when you intentionally want no positional encoding (pass `positional_encoding="none"` or `"nope"` to the transformer).

**Use for:** sets of objects (bounding boxes, molecules, graph nodes), permutation-invariant tasks.

In [3]:
set_tok = SetTokenizer(input_dim=8, embedding_dim=64)

# (batch, set_size, features_per_element)
# Real: bounding boxes -> (B, n_boxes, 5),  molecule atoms -> (B, n_atoms, 4)
x      = torch.randn(2, 20, 8)
tokens = set_tok(x)
print(f"Set tokens: {x.shape} -> {tokens.shape}")

Set tokens: torch.Size([2, 20, 8]) -> torch.Size([2, 20, 64])


### 1.3 `DiscreteTokenTokenizer`

Standard learnable embedding table for integer token IDs (`nn.Embedding`).

**Integrates into:** `TokenToTokenTransformer`, `TokenToClassTransformer` etc. use `ContinuousInputTokenizer`  -  if your input is discrete IDs, use this **before** feeding into a `TransformerStack` manually, or use `TokenToTokenTransformer` with the matching `input_dim=embedding_dim` (after embedding).

**Use for:** NLP (BPE subwords), discrete audio tokens (EnCodec), VQVAE image codes.

In [4]:
tokenizer = DiscreteTokenTokenizer(
    vocab_size=8192, embedding_dim=128, padding_idx=0,
)

# (batch, sequence_length)  -  integer IDs in [0, vocab_size)
# Real: BPE tokens -> (B, 128),  VQVAE image codes -> (B, 256)  (16x16 flattened)
ids    = torch.randint(1, 8192, (2, 32))   # avoid 0 (padding_idx)
tokens = tokenizer(ids)
print(f"{ids.shape} -> {tokens.shape}")

torch.Size([2, 32]) -> torch.Size([2, 32, 128])


### 1.4 `PatchTokenizerND`  -  1D / 2D / 3D

Splits a spatial tensor into non-overlapping patches and projects each patch to `embedding_dim` via a strided convolution.

**Integrates into:** used as the input stage of `PatchTransformerND`. Use it standalone when you want to patchify before a custom `TransformerStack` or when you need the `grid_shape` for a `PatchDecoderND` later.

Returns `(tokens, grid_shape)`  -  save `grid_shape` to pass to `PatchDecoderND` for reconstruction.

**Use for:** ViT-style image models, audio spectrogram patch encoders, spatiotemporal video tokens.

In [5]:
# 1D patches (waveform segments)
ptok_1d = PatchTokenizerND(spatial_dim=1, in_channels=1, embedding_dim=64, patch_size=8)
x_1d    = torch.randn(2, 1, 256)
# Real: audio (B, 1, 16000) with patch_size=160 -> 100 patches of 10 ms each
tokens_1d, grid_1d = ptok_1d(x_1d)
print(f"1D: {x_1d.shape} -> tokens={tokens_1d.shape}, grid={grid_1d}")

# 2D patches (ViT-style)
ptok_2d = PatchTokenizerND(spatial_dim=2, in_channels=3, embedding_dim=256, patch_size=4)
x_2d    = torch.randn(2, 3, 32, 32)
# Real: ImageNet (B, 3, 224, 224) with patch_size=16 -> 196 patches (14x14)
tokens_2d, grid_2d = ptok_2d(x_2d)
print(f"2D: {x_2d.shape} -> tokens={tokens_2d.shape}, grid={grid_2d}")

# 3D patches (volume / video)
ptok_3d = PatchTokenizerND(spatial_dim=3, in_channels=1, embedding_dim=128, patch_size=4)
x_3d    = torch.randn(2, 1, 16, 16, 16)
# Real: video (B, 3, 8, 224, 224) with patch_size=(2,16,16) -> spatiotemporal tokens
tokens_3d, grid_3d = ptok_3d(x_3d)
print(f"3D: {x_3d.shape} -> tokens={tokens_3d.shape}, grid={grid_3d}")

1D: torch.Size([2, 1, 256]) -> tokens=torch.Size([2, 32, 64]), grid=(32,)
2D: torch.Size([2, 3, 32, 32]) -> tokens=torch.Size([2, 64, 256]), grid=(8, 8)
3D: torch.Size([2, 1, 16, 16, 16]) -> tokens=torch.Size([2, 64, 128]), grid=(4, 4, 4)


---
## 2. Attention Modules

These are the low-level attention operations. You rarely need them directly  -  `TransformerBlock` composes them with normalisation and a feed-forward layer. Use these standalone only when building a custom architecture that doesn't fit the standard pre-norm block pattern.

### 2.1 `MultiHeadSelfAttention`

Standard multi-head self-attention. Optionally causal or RoPE-encoded.

**Integrates into:** the self-attention sub-layer of `TransformerBlock`.

**Use standalone when:** building a custom attention layer that deviates from the standard pre-norm + FFN pattern  -  e.g. a shared-weight attention across layers, or an attention-only network.

| Option | When to use |
|---|---|
| `causal=True` | Autoregressive generation  -  token `t` cannot see `t+1, t+2, ...` |
| `positional_encoding="rope"` | Long sequences where absolute position biases are undesirable; RoPE is relative |
| `positional_encoding="none"` | PE handled upstream (by the tokenizer or model entry)  -  the common case |

In [6]:
tokens = torch.randn(2, 32, 128)   # (B, T, D)
mask   = torch.ones(2, 32, dtype=torch.bool)
mask[0, 20:] = False   # sample 0: 20 real tokens, 12 padding

# Bidirectional (encoder)
attn = MultiHeadSelfAttention(embedding_dim=128, num_heads=4, causal=False)
print(f"Bidirectional: {attn(tokens, mask=mask).shape}")

# Causal (autoregressive decoder)
attn_causal = MultiHeadSelfAttention(embedding_dim=128, num_heads=4, causal=True)
print(f"Causal: {attn_causal(tokens).shape}")

# RoPE (relative position encoding  -  no max_length constraint)
attn_rope = MultiHeadSelfAttention(embedding_dim=128, num_heads=4, positional_encoding="rope")
print(f"RoPE: {attn_rope(tokens).shape}")

Bidirectional: torch.Size([2, 32, 128])
Causal: torch.Size([2, 32, 128])
RoPE: torch.Size([2, 32, 128])


### 2.2 `MultiHeadCrossAttention`

Queries come from one sequence, keys/values from another.

**Integrates into:** the cross-attention sub-layer of `TransformerBlock` (when `use_cross_attention=True`). Also used inside `SpatialAttentionBlock` in the UNet module for text-guided spatial attention.

**Use standalone when:** building a custom encoder-decoder where you want cross-attention without the full TransformerBlock overhead  -  e.g. a single-layer cross-attention fusion between two modalities.

**Use for:** decoder attending to encoder output, image features attending to text tokens, any query-context fusion.

In [7]:
cross_attn = MultiHeadCrossAttention(
    query_dim=128, context_dim=256, num_heads=4,
)

queries = torch.randn(2, 16, 128)   # (B, T_query, query_dim)
context = torch.randn(2, 64, 256)   # (B, T_context, context_dim)
# Real: queries = image patches (B, 196, 768),  context = text tokens (B, 77, 512)

out = cross_attn(queries, context)
print(f"Cross-attn: q={queries.shape}, ctx={context.shape} -> {out.shape}")

# With context mask (True = valid token)
ctx_mask = torch.ones(2, 64, dtype=torch.bool)
ctx_mask[1, 32:] = False
out_masked = cross_attn(queries, context, context_mask=ctx_mask)
print(f"Masked: {out_masked.shape}")

Cross-attn: q=torch.Size([2, 16, 128]), ctx=torch.Size([2, 64, 256]) -> torch.Size([2, 16, 128])
Masked: torch.Size([2, 16, 128])


---
## 3. `FeedForward`, `TransformerBlock`, `TransformerStack`

These form the standard transformer block hierarchy. You almost never construct `FeedForward` or `TransformerBlock` directly  -  instead, use `TransformerStack` or a full model. Build them manually only for non-standard architectures.

### 3.1 `FeedForward`

The two-layer MLP inside each transformer block. Hidden dim = `embedding_dim x mlp_ratio`.

**Integrates into:** `TransformerBlock`  -  every block contains one `FeedForward`.

**Use standalone when:** you need the FFN sub-layer without the surrounding attention + normalisation  -  rare.

In [8]:
ff     = FeedForward(embedding_dim=128, hidden_dim=512, dropout=0.0)
tokens = torch.randn(2, 32, 128)
print(f"FeedForward: {tokens.shape} -> {ff(tokens).shape}")

FeedForward: torch.Size([2, 32, 128]) -> torch.Size([2, 32, 128])


### 3.2 `TransformerBlock`

Pre-norm block: `LayerNorm -> Attention -> residual -> LayerNorm -> FFN -> residual`.

**Integrates into:** `TransformerStack`  -  a stack of identical `TransformerBlock` layers.

**Use standalone when:** you need blocks with heterogeneous configurations across layers (e.g. alternating causal and non-causal layers, or different cross-attention dims per layer). For uniform stacks, use `TransformerStack`.

| Variant | When to use |
|---|---|
| `use_cross_attention=False` (default) | Encoder, bidirectional sequence model |
| `use_cross_attention=True` | Decoder, encoder-decoder, text-conditioned model |
| `causal=True` | Autoregressive generation |
| `norm_type="rms"` | LLaMA / Mistral style  -  slightly faster, no bias |

In [9]:
tokens = torch.randn(2, 32, 128)

# Standard encoder block
block = TransformerBlock(embedding_dim=128, num_heads=4, norm_type="layer")
print(f"\nEncoder block: {block(tokens).shape}")
print(block)
print("="*20)

# RMS norm (LLaMA-style)
block_rms = TransformerBlock(embedding_dim=128, num_heads=4, norm_type="rms")
print(f"\nRMS-norm block: {block_rms(tokens).shape}")
print(block_rms)

# Causal block
block_causal = TransformerBlock(embedding_dim=128, num_heads=4, causal=True)
print(f"\nCausal block: {block_causal(tokens).shape}")
print(block_causal)
print("="*20)

# Cross-attention block (encoder-decoder)
block_cross = TransformerBlock(
    embedding_dim=128, num_heads=4,
    use_cross_attention=True, cross_attention_dim=256,
)
context = torch.randn(2, 20, 256)
print(f"\nCross-attn block: {block_cross(tokens, context=context).shape}")
print(block_cross)


Encoder block: torch.Size([2, 32, 128])
TransformerBlock(
  (self_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (self_attention): MultiHeadSelfAttention(
    (qkv_projection): Linear(in_features=128, out_features=384, bias=True)
    (output_projection): Linear(in_features=128, out_features=128, bias=True)
  )
  (ff_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (feed_forward): FeedForward(
    (net): Sequential(
      (0): Linear(in_features=128, out_features=512, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.0, inplace=False)
      (3): Linear(in_features=512, out_features=128, bias=True)
      (4): Dropout(p=0.0, inplace=False)
    )
  )
)

RMS-norm block: torch.Size([2, 32, 128])
TransformerBlock(
  (self_norm): RMSNorm((128,), eps=None, elementwise_affine=True)
  (self_attention): MultiHeadSelfAttention(
    (qkv_projection): Linear(in_features=128, out_features=384, bias=True)
    (output_projection): 

### 3.3 `TransformerStack`

Stacks `depth` identical `TransformerBlock` layers with an optional final normalisation.

**Integrates into:** the core of all full transformer models (`TokenToTokenTransformer`, `ConditionedTokenTransformer`, `PatchTransformerND`, etc.).

**Use standalone when:** you want to assemble a custom transformer with your own tokenizer and head  -  e.g. a multi-modal encoder that uses different tokenizers per modality but shares a transformer trunk.

> If you find yourself writing `TransformerStack` + a tokenizer + a head, check whether one of the full model classes already covers your use case.

In [10]:
tokens = torch.randn(2, 32, 128)

# Encoder stack  -  bidirectional, final LayerNorm
encoder = TransformerStack(
    embedding_dim=128, depth=6, num_heads=4,
    mlp_ratio=4.0, dropout=0.1, causal=False, final_norm=True,
)
print(f"Encoder stack (depth=6): {encoder(tokens).shape}")
print(encoder)
print("="*20)

# Decoder stack  -  causal + cross-attention
decoder = TransformerStack(
    embedding_dim=128, depth=6, num_heads=4, causal=True,
    use_cross_attention=True, cross_attention_dim=256, final_norm=True,
)
dec_tokens = torch.randn(2, 16, 128)
enc_output = torch.randn(2, 32, 256)
print(f"Decoder stack (depth=6): {decoder(dec_tokens, context=enc_output).shape}")
print(decoder)
print("="*20)

Encoder stack (depth=6): torch.Size([2, 32, 128])
TransformerStack(
  (blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (self_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (self_attention): MultiHeadSelfAttention(
        (qkv_projection): Linear(in_features=128, out_features=384, bias=True)
        (output_projection): Linear(in_features=128, out_features=128, bias=True)
      )
      (ff_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (feed_forward): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=512, out_features=128, bias=True)
          (4): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
)
Decoder stack (depth=6): torch.Size([2, 16, 128])
Tra

---
## 4. Pooling

`TokenPooling` collapses `(B, T, D)` -> `(B, D)` to produce a single summary vector.

**Integrates into:** `PooledHead` (and its aliases `ClassificationHead`, `RegressionHead`) and the full models `TokenToVectorTransformer` and `TokenToClassTransformer`.

**Use standalone when:** building a custom head that first pools tokens then applies an MLP.

| Mode | How it pools | Best for |
|---|---|---|
| `"mean"` | Masked average over valid tokens | General-purpose encoder (BERT-style) |
| `"max"` | Max over valid tokens | Sparse features, retrieval |
| `"cls"` | Returns token 0 | Add a learnable `[CLS]` token before the sequence |
| `"last"` | Last valid token | Causal/autoregressive models (GPT-style) |
| `"attention"` | Learned weighted average | When different tokens have variable relevance |

In [11]:
tokens = torch.randn(2, 32, 128)
mask   = torch.ones(2, 32, dtype=torch.bool)
mask[1, 20:] = False

for mode in ["mean", "max", "cls", "last", "attention"]:
    pooler = TokenPooling(mode=mode, embedding_dim=128 if mode == "attention" else None)
    pooled = pooler(tokens, mask=mask)
    print(f"mode='{mode}': {tokens.shape} -> {pooled.shape}")

mode='mean': torch.Size([2, 32, 128]) -> torch.Size([2, 128])
mode='max': torch.Size([2, 32, 128]) -> torch.Size([2, 128])
mode='cls': torch.Size([2, 32, 128]) -> torch.Size([2, 128])
mode='last': torch.Size([2, 32, 128]) -> torch.Size([2, 128])
mode='attention': torch.Size([2, 32, 128]) -> torch.Size([2, 128])


---
## 5. Heads

Heads map processed token tensors to task outputs. You typically get these for free by using a full model (`TokenToClassTransformer` etc.), but they are composable if you need a custom trunk.

### 5.1 `TokenwiseHead` / `TokenDecoder`

Applies the same MLP independently to each token. `TokenDecoder` is an alias.

**Integrates into:** `QuerySetDecoder` uses `TokenwiseHead` internally to project each query slot to the output dimension.

**Use for:** token-level prediction (NER, sequence labelling, per-pixel prediction, optical flow per patch, per-token regression).

In [12]:
head   = TokenwiseHead(embedding_dim=128, output_dim=16, hidden_dim=64, num_layers=2)
tokens = torch.randn(2, 32, 128)   # (B, T, D)
# Real output: NER tags (B, T, n_tags),  per-patch flow (B, n_patches, 2)
print(f"TokenwiseHead: {tokens.shape} -> {head(tokens).shape}")

TokenwiseHead: torch.Size([2, 32, 128]) -> torch.Size([2, 32, 16])


### 5.2 `PooledHead`, `ClassificationHead`, `RegressionHead`

Pools the token sequence (via `TokenPooling`) then applies an MLP.
`ClassificationHead` and `RegressionHead` are aliases  -  semantically annotate your intent.

**Integrates into:** `TokenToVectorTransformer` and `TokenToClassTransformer` use this pattern internally.

**Use for:** sentence/document classification, audio clip classification, sequence-level regression.

In [13]:
tokens = torch.randn(2, 32, 128)
mask   = torch.ones(2, 32, dtype=torch.bool)

cls_head = ClassificationHead(embedding_dim=128, output_dim=5, pooling="mean", num_layers=2)
print(f"ClassificationHead: {cls_head(tokens, mask=mask).shape}")

reg_head = RegressionHead(embedding_dim=128, output_dim=1, pooling="cls")
print(f"RegressionHead: {reg_head(tokens, mask=mask).shape}")

ClassificationHead: torch.Size([2, 5])
RegressionHead: torch.Size([2, 1])


---
## 6. Conditioning & Time Embeddings

These modules form the conditioning pipeline used by `ConditionedTokenTransformer`:

```
SinusoidalTimeEmbedding  ->  TimeEmbeddingMLP  ->  TransformerConditioningBuilder
                                                          v
                                               ConditionedTokenTransformer
                                               (adds conditioning vec to all tokens)
```

Use these standalone only when building a custom conditioned model outside the full model classes.

### 6.1 `SinusoidalTimeEmbedding`

Fixed (no-param) mapping from a scalar timestep to a high-dimensional vector.

**Integrates into:** `TimeEmbeddingMLP`  -  the sinusoidal vector is the base that the MLP refines.

**Use standalone when:** you need a raw sinusoidal time feature (e.g. for an external conditioning pipeline). In most cases, use `TimeEmbeddingMLP` instead as it also adds a learnable projection.

In [14]:
sin_emb = SinusoidalTimeEmbedding(embedding_dim=128)
time    = torch.tensor([0.1, 0.5])   # continuous diffusion timestep
emb     = sin_emb(time)
print(f"SinusoidalTimeEmbedding: {time.shape} -> {emb.shape}")

SinusoidalTimeEmbedding: torch.Size([2]) -> torch.Size([2, 128])


### 6.2 `TimeEmbeddingMLP`

Adds a 2-layer MLP on top of sinusoidal or learned base, mapping to `embedding_dim`.

**Integrates into:** `TransformerConditioningBuilder`  -  it is the time branch inside the conditioning builder.

**Use standalone when:** you need a ready-to-use time embedding in a custom diffusion model that manages its own conditioning.

In [15]:
time = torch.rand(2)

t_sin = TimeEmbeddingMLP(embedding_dim=128, embedding_type="sinusoidal")
print(f"Sinusoidal: {t_sin(time).shape}")

t_learned = TimeEmbeddingMLP(embedding_dim=128, embedding_type="learned")
print(f"Learned: {t_learned(time).shape}")

Sinusoidal: torch.Size([2, 128])
Learned: torch.Size([2, 128])


### 6.3 `TransformerConditioningBuilder`

Combines time + class label + global context into a single `(B, embedding_dim)` vector by summing the active branches.

**Integrates into:** `ConditionedTokenTransformer` uses this internally. The output vector is broadcast-added to all tokens before/inside the stack.

**Use standalone when:** assembling a custom conditioned transformer with `TransformerStack` that needs the same conditioning logic.

In [16]:
builder = TransformerConditioningBuilder(
    embedding_dim=128,
    time_conditioning=True,
    num_classes=10,
    global_context_dim=64,
)

B = 2
cond = builder(
    batch_size=B, device=torch.device("cpu"), dtype=torch.float32,
    time=torch.rand(B),
    class_labels=torch.randint(0, 10, (B,)),
    global_context=torch.randn(B, 64),
)
print(f"Conditioning vector: {cond.shape}")

Conditioning vector: torch.Size([2, 128])


### 6.4 `ConditionTokenProjector`

Projects a single global context vector into one or more **condition tokens** that can be prepended to the sequence.

**Integrates into:** nothing in the library uses this automatically  -  it is a standalone utility.

**Use when:** you want context tokens that the model can attend to (prepend-style conditioning), rather than adding a scalar offset to all tokens. Analogous to T5's encoder-decoder prefix or CLIPSeg's conditioning tokens.

In [17]:
projector   = ConditionTokenProjector(context_dim=256, embedding_dim=128, num_tokens=4)
ctx         = torch.randn(2, 256)   # e.g. CLIP image embedding
cond_tokens = projector(ctx)
print(f"Condition tokens: {ctx.shape} -> {cond_tokens.shape}")   # (2, 4, 128)

# Typical usage: prepend to input tokens before TransformerStack
input_tokens   = torch.randn(2, 32, 128)
combined       = torch.cat([cond_tokens, input_tokens], dim=1)
print(f"Combined: {combined.shape}")   # (2, 36, 128)

Condition tokens: torch.Size([2, 256]) -> torch.Size([2, 4, 128])
Combined: torch.Size([2, 36, 128])


---
## 7. Decoders

### 7.1 `PatchDecoderND`

Reconstructs a spatial grid from patch tokens using transposed convolution  -  the inverse of `PatchTokenizerND`.

**Integrates into:** used as the output stage of `PatchTransformerND(output_mode="grid")`.

**Use standalone when:** you want to decode transformer-processed patch tokens back to spatial coordinates in a custom pipeline  -  e.g. a MAE (Masked Autoencoder) reconstruction head.

**Use for:** dense prediction from ViT, image reconstruction, output heads for spatial diffusion transformers.

In [18]:
# Round-trip: patchify -> process -> unpatchify
ptok   = PatchTokenizerND(spatial_dim=2, in_channels=3, embedding_dim=128, patch_size=4)
x      = torch.randn(2, 3, 32, 32)
tokens, grid_shape = ptok(x)
print(f"Patchified: {x.shape} -> {tokens.shape}, grid={grid_shape}")

# ... transformer processing goes here ...

decoder = PatchDecoderND(input_dim=2, embedding_dim=128, out_channels=3, patch_size=4)
recon   = decoder(tokens, grid_shape)
print(f"Decoded: {tokens.shape} + grid={grid_shape} -> {recon.shape}")

Patchified: torch.Size([2, 3, 32, 32]) -> torch.Size([2, 64, 128]), grid=(8, 8)
Decoded: torch.Size([2, 64, 128]) + grid=(8, 8) -> torch.Size([2, 3, 32, 32])


### 7.2 `QuerySetDecoder`

Decodes a global latent vector into a fixed-size **set of output slots** using learned query vectors (DETR-style).

**Integrates into:** nothing in the library uses this automatically  -  it is a standalone structured-output decoder.

**Use for:** object detection (each query -> one predicted box + class), slot-based decomposition, fixed-size output generation from a variable-length encoding.

In [19]:
decoder = QuerySetDecoder(
    latent_dim=128, output_dim=6, num_queries=10, query_dim=64,
    # output_dim=6: e.g. (x, y, w, h) + 2 class scores
)

latent = torch.randn(2, 128)   # pooled representation from encoder
# Real: DETR: latent = Transformer encoder CLS token (B, 256)
out    = decoder(latent)
print(f"QuerySetDecoder: {latent.shape} -> {out.shape}")   # (2, 10, 6)

QuerySetDecoder: torch.Size([2, 128]) -> torch.Size([2, 10, 6])


---
## 8. Full Transformer Models

These are the recommended entry points for end users. They wire together tokenizer + positional encoding + `TransformerStack` + appropriate head/decoder into one clean module.

**When to use each:**
| Model | Use when |
|---|---|
| `TokenToTokenTransformer` | Output is same-length token sequence (denoising, seq2seq without vocab) |
| `TokenToVectorTransformer` | Output is a single embedding vector (retrieval, downstream regression) |
| `TokenToClassTransformer` | Output is class logits (classification) |
| `ConditionedTokenTransformer` | Need time/class/cross-attention conditioning (diffusion, controlled generation) |
| `PatchTransformerND` | Input is a spatial grid (ViT for images / volumes / signals) |

### 8.1 `TokenToTokenTransformer`

Maps token sequence -> token sequence of the same length. Input and output can have different feature dims.

**Applications:** denoising (noisy tokens -> clean tokens), feature sequence refinement, non-autoregressive sequence editing, autoregressive next-token prediction (with `causal=True`).

**Positional encoding options:**
| `positional_encoding` | When to use |
|---|---|
| `"learned"` | Fixed-length sequences; most common for classification/generation |
| `"sinusoidal"` | Fixed pattern; transferable to unseen lengths (within reason) |
| `"rope"` | No `max_length` constraint; best for variable-length / long sequences |
| `"none"` / `"nope"` | Input already carries position (e.g. set inputs, patches with 2D PE) |

In [20]:
x    = torch.randn(2, 32, 32)
mask = torch.ones(2, 32, dtype=torch.bool)

# Bidirectional  -  sequence refinement
model = TokenToTokenTransformer(
    input_dim=32, output_dim=16, embedding_dim=128,
    depth=4, num_heads=4, max_length=64, positional_encoding="learned",
)
print(f"Tok->Tok (learned PE): {x.shape} -> {model(x, mask=mask).shape}")

# Causal  -  autoregressive token prediction
model_causal = TokenToTokenTransformer(
    input_dim=32, output_dim=32, embedding_dim=128,
    depth=4, num_heads=4, positional_encoding="sinusoidal", causal=True,
)
print(f"Tok->Tok (causal): {model_causal(x).shape}")

# RoPE  -  no max_length, robust to variable sequence lengths
model_rope = TokenToTokenTransformer(
    input_dim=32, output_dim=32, embedding_dim=128,
    depth=4, num_heads=4, positional_encoding="rope",
)
print(f"Tok->Tok (RoPE): {model_rope(x).shape}")

Tok->Tok (learned PE): torch.Size([2, 32, 32]) -> torch.Size([2, 32, 16])
Tok->Tok (causal): torch.Size([2, 32, 32])
Tok->Tok (RoPE): torch.Size([2, 32, 32])


### 8.2 `TokenToVectorTransformer`

Encodes a token sequence to a single pooled vector.

**Applications:** sentence / document embedding (BERT-style), audio clip embedding, contrastive learning encoders (CLIP, SimCSE), feature extraction for downstream tasks.

**How to pick the pooling mode:**
- `"mean"`  -  safe default for variable-length inputs
- `"cls"`  -  prepend a special token; lets the model learn what to summarise
- `"last"`  -  best for causal/autoregressive encoders
- `"max"`  -  keyword spotting, retrieval with sparse signal

In [21]:
x    = torch.randn(2, 32, 32)
mask = torch.ones(2, 32, dtype=torch.bool)
mask[1, 20:] = False

for pool in ["mean", "max", "cls", "last"]:
    model = TokenToVectorTransformer(
        input_dim=32, output_dim=64, embedding_dim=128,
        depth=3, num_heads=4, positional_encoding="sinusoidal", pooling=pool,
    )
    print(f"pooling='{pool}': {model(x, mask=mask).shape}")

pooling='mean': torch.Size([2, 64])
pooling='max': torch.Size([2, 64])
pooling='cls': torch.Size([2, 64])
pooling='last': torch.Size([2, 64])


### 8.3 `TokenToClassTransformer`

Convenience wrapper: same as `TokenToVectorTransformer` with `output_dim = num_classes`.

**Applications:** text classification (sentiment, topic), audio event classification, activity recognition from sensor sequences, document categorisation.

**When to use this vs. `TokenToVectorTransformer`:** if you need class logits directly, use this. If you need a shared encoder that feeds multiple heads or a contrastive objective, use `TokenToVectorTransformer` and attach heads separately.

In [22]:
model  = TokenToClassTransformer(
    input_dim=16, num_classes=10, embedding_dim=128,
    depth=4, num_heads=4, max_length=128,
    positional_encoding="learned", pooling="mean",
)
x      = torch.randn(2, 32, 16)
logits = model(x)
print(f"Classification: {x.shape} -> {logits.shape}")

Classification: torch.Size([2, 32, 16]) -> torch.Size([2, 10])


### 8.4 `ConditionedTokenTransformer`

Extends `TokenToTokenTransformer` with all four conditioning sources. Any subset can be active.

**Applications:** diffusion transformer (DiT-style), text/class-conditioned sequence generation, score networks with explicit conditioning signals.

**How conditioning works:** time + class + global_context are combined by `TransformerConditioningBuilder` into a single vector that is **added to all tokens** before each stack layer. Cross-context uses `MultiHeadCrossAttention` inside each `TransformerBlock` with `use_cross_attention=True`.

In [23]:
x = torch.randn(2, 32, 16)

# Time only (diffusion score network)
model_t = ConditionedTokenTransformer(
    input_dim=16, output_dim=16, embedding_dim=128, depth=4, num_heads=4,
    time_conditioning=True, positional_encoding="sinusoidal",
)
print(f"Time conditioning: {model_t(x, time=torch.rand(2)).shape}")

# Class conditioning (classifier-free guidance)
model_cls = ConditionedTokenTransformer(
    input_dim=16, output_dim=16, embedding_dim=128, depth=4, num_heads=4,
    num_classes=10, class_dropout_prob=0.1, positional_encoding="sinusoidal",
)
print(f"Class conditioning: {model_cls(x, class_labels=torch.randint(0, 10, (2,))).shape}")

# Global context (e.g. CLIP pooled embedding)
model_ctx = ConditionedTokenTransformer(
    input_dim=16, output_dim=16, embedding_dim=128, depth=4, num_heads=4,
    global_context_dim=64, positional_encoding="sinusoidal",
)
print(f"Global context: {model_ctx(x, global_context=torch.randn(2, 64)).shape}")

# Cross-attention (attend to token sequence, e.g. BERT tokens)
model_cross = ConditionedTokenTransformer(
    input_dim=16, output_dim=16, embedding_dim=128, depth=4, num_heads=4,
    cross_attention_dim=256, positional_encoding="sinusoidal",
)
cross_ctx = torch.randn(2, 20, 256)
ctx_mask  = torch.ones(2, 20, dtype=torch.bool)
print(f"Cross-attn: {model_cross(x, cross_context=cross_ctx, cross_context_mask=ctx_mask).shape}")

# All four combined
model_full = ConditionedTokenTransformer(
    input_dim=16, output_dim=16, embedding_dim=128, depth=4, num_heads=4,
    time_conditioning=True, num_classes=10, global_context_dim=64, cross_attention_dim=256,
    positional_encoding="sinusoidal",
)
out = model_full(
    x, time=torch.rand(2), class_labels=torch.randint(0, 10, (2,)),
    global_context=torch.randn(2, 64),
    cross_context=cross_ctx, cross_context_mask=ctx_mask,
)
print(f"Full conditioning: {out.shape}")

Time conditioning: torch.Size([2, 32, 16])
Class conditioning: torch.Size([2, 32, 16])
Global context: torch.Size([2, 32, 16])
Cross-attn: torch.Size([2, 32, 16])
Full conditioning: torch.Size([2, 32, 16])


### 8.5 `PatchTransformerND`  -  Vision Transformer for 1D / 2D / 3D

End-to-end ViT that handles patchification (`PatchTokenizerND`) + positional encoding + `TransformerStack` + output decoding in one model.

**Applications:** image classification / embedding, dense prediction (segmentation, super-resolution), volumetric analysis, 1D signal encoding.

**Output modes:**
| `output_mode` | Returns | Use for |
|---|---|---|
| `"grid"` | `(B, out_channels, *spatial)`  -  same shape as input | Dense prediction  -  `PatchDecoderND` is used internally |
| `"tokens"` | `(B, n_patches, out_channels)` | Downstream token processing, e.g. cross-attention with another model |
| `"vector"` | `(B, vector_output_dim)` | Global embedding  -  classification, retrieval, contrastive |

In [24]:
x = torch.randn(2, 3, 32, 32)

# Grid output  -  dense prediction (segmentation, MAE reconstruction)
vit_grid = PatchTransformerND(
    input_dim=2, in_channels=3, out_channels=3, patch_size=4,
    embedding_dim=128, depth=4, num_heads=4,
    output_mode="grid", positional_encoding="sinusoidal",
)
print(f"2D grid: {x.shape} -> {vit_grid(x).shape}")

# Token output  -  downstream processing (feed to another transformer, cross-attention)
vit_tok = PatchTransformerND(
    input_dim=2, in_channels=3, out_channels=64, patch_size=4,
    embedding_dim=128, depth=4, num_heads=4, output_mode="tokens",
)
print(f"2D tokens: {x.shape} -> {vit_tok(x).shape}")

# Vector output  -  image embedding for classification or retrieval
vit_vec = PatchTransformerND(
    input_dim=2, in_channels=3, out_channels=None, patch_size=4,
    embedding_dim=128, depth=4, num_heads=4,
    output_mode="vector", vector_output_dim=256, pooling="mean",
)
print(f"2D vector: {x.shape} -> {vit_vec(x).shape}")

# 1D ViT (waveform / spectrogram encoder)
vit_1d = PatchTransformerND(
    input_dim=1, in_channels=1, out_channels=1, patch_size=8,
    embedding_dim=64, depth=3, num_heads=4, output_mode="grid",
)
x_1d = torch.randn(2, 1, 256)
print(f"1D grid: {x_1d.shape} -> {vit_1d(x_1d).shape}")

# 3D ViT (volumetric embedding)
vit_3d = PatchTransformerND(
    input_dim=3, in_channels=1, out_channels=None, patch_size=4,
    embedding_dim=64, depth=3, num_heads=4,
    output_mode="vector", vector_output_dim=64,
)
x_3d = torch.randn(1, 1, 16, 16, 16)
print(f"3D vector: {x_3d.shape} -> {vit_3d(x_3d).shape}")

2D grid: torch.Size([2, 3, 32, 32]) -> torch.Size([2, 3, 32, 32])
2D tokens: torch.Size([2, 3, 32, 32]) -> torch.Size([2, 64, 64])
2D vector: torch.Size([2, 3, 32, 32]) -> torch.Size([2, 256])
1D grid: torch.Size([2, 1, 256]) -> torch.Size([2, 1, 256])
3D vector: torch.Size([1, 1, 16, 16, 16]) -> torch.Size([1, 64])


---
## 9. Use Case Recipes

These recipes show how to wire the presets together for real tasks. Each is a minimal, runnable example  -  copy, adjust dims, and train.

| Task | Key classes |
|---|---|
| Variable-length seq -> variable-length seq (translation, transcription) | encoder `TokenToTokenTransformer` + decoder `TransformerStack` (causal + cross-attn) |
| Diffusion / score network | `ConditionedTokenTransformer` |
| Autoregressive language model | `DiscreteTokenTokenizer` + `TokenToTokenTransformer(causal=True)` |
| Time-series / audio classification | `TokenToClassTransformer` |
| Contrastive dual encoder (CLIP-style) | two `TokenToVectorTransformer` + cosine loss |
| Object detection (DETR-style) | `PatchTransformerND(output_mode="tokens")` + `QuerySetDecoder` |

### 9.1 Variable-Length Seq -> Variable-Length Seq (Encoder-Decoder)

Translation, summarisation, transcription  -  tasks where the output length differs from the input.

**Stack:**
```
src -> ContinuousInputTokenizer -> TransformerStack (bidirectional)  <- encoder
tgt -> ContinuousInputTokenizer -> TransformerStack (causal + cross-attn -> encoder output) -> TokenwiseHead  <- decoder
```

The encoder reads the full source in parallel. The decoder generates one token at a time, attending to the encoder output via cross-attention at every layer.

> For discrete text (BPE tokens), swap `ContinuousInputTokenizer` for `DiscreteTokenTokenizer` and point the output head at `vocab_size`.

In [25]:
import torch.nn as nn

EMB = 128
SRC_DIM, TGT_DIM, OUT_DIM = 32, 32, 32

# Encoder: bidirectional, reads the full source sequence
src_tokenizer = ContinuousInputTokenizer(input_dim=SRC_DIM, embedding_dim=EMB)
encoder_stack = TransformerStack(
    embedding_dim=EMB, depth=4, num_heads=4, causal=False, final_norm=True,
)

# Decoder: causal self-attention + cross-attention to encoder output
tgt_tokenizer = ContinuousInputTokenizer(input_dim=TGT_DIM, embedding_dim=EMB)
decoder_stack = TransformerStack(
    embedding_dim=EMB, depth=4, num_heads=4,
    causal=True,                        # can only see past target tokens
    use_cross_attention=True,           # attends to encoder output
    cross_attention_dim=EMB,
    final_norm=True,
)
output_head = TokenwiseHead(embedding_dim=EMB, output_dim=OUT_DIM)

# --- forward pass (teacher-forced, training) ---
B = 2
src = torch.randn(B, 20, SRC_DIM)   # variable-length source  (B, T_src, src_dim)
tgt = torch.randn(B, 15, TGT_DIM)   # variable-length target  (B, T_tgt, tgt_dim)

enc_tokens = encoder_stack(src_tokenizer(src))          # (B, T_src, EMB)
dec_tokens = decoder_stack(
    tgt_tokenizer(tgt),
    context=enc_tokens,                                 # cross-attend to encoder
)                                                       # (B, T_tgt, EMB)
out = output_head(dec_tokens)                           # (B, T_tgt, OUT_DIM)

print(f"src={src.shape}  tgt={tgt.shape}")
print(f"encoder output : {enc_tokens.shape}")
print(f"decoder output : {dec_tokens.shape}")
print(f"prediction     : {out.shape}   <- same length as target, not source")

src=torch.Size([2, 20, 32])  tgt=torch.Size([2, 15, 32])
encoder output : torch.Size([2, 20, 128])
decoder output : torch.Size([2, 15, 128])
prediction     : torch.Size([2, 15, 32])   <- same length as target, not source


### 9.2 Diffusion Score Network (DiT-style)

A denoising transformer that predicts the noise in a noisy token sequence, conditioned on the diffusion timestep and optionally a class label.

**Stack:** `ConditionedTokenTransformer` with `time_conditioning=True`. The timestep is embedded by `SinusoidalTimeEmbedding -> TimeEmbeddingMLP -> TransformerConditioningBuilder` and broadcast-added to every token before each layer.

**Training loop sketch:**
```python
noise = torch.randn_like(x0)
t     = torch.randint(0, T, (B,))
xt    = scheduler.add_noise(x0, noise, t)
pred  = model(xt, time=t / T)
loss  = F.mse_loss(pred, noise)
```

In [26]:
import torch.nn.functional as F

# Unconditional (time only)  -  e.g. audio / motion denoising
score_net = ConditionedTokenTransformer(
    input_dim=16, output_dim=16, embedding_dim=256,
    depth=6, num_heads=8,
    time_conditioning=True,
    positional_encoding="sinusoidal",
)

# Class-conditional (time + class)  -  classifier-free guidance ready
score_net_cfg = ConditionedTokenTransformer(
    input_dim=16, output_dim=16, embedding_dim=256,
    depth=6, num_heads=8,
    time_conditioning=True,
    num_classes=10,
    class_dropout_prob=0.1,       # randomly drop class during training for CFG
    positional_encoding="sinusoidal",
)

B, T, D = 2, 32, 16
x0    = torch.randn(B, T, D)
noise = torch.randn_like(x0)
t_int = torch.randint(0, 1000, (B,))
t_norm = t_int.float() / 1000.0   # normalise to [0, 1]

# Simulate a noisy sample (linear interpolation, not a real scheduler)
xt = 0.9 * x0 + 0.1 * noise

pred_uncond = score_net(xt, time=t_norm)
pred_cond   = score_net_cfg(xt, time=t_norm, class_labels=torch.randint(0, 10, (B,)))

loss_uncond = F.mse_loss(pred_uncond, noise)
loss_cond   = F.mse_loss(pred_cond, noise)

print(f"Input (noisy)  : {xt.shape}")
print(f"Predicted noise: {pred_uncond.shape}")
print(f"Uncond loss    : {loss_uncond.item():.4f}")
print(f"Cond loss      : {loss_cond.item():.4f}")

Input (noisy)  : torch.Size([2, 32, 16])
Predicted noise: torch.Size([2, 32, 16])
Uncond loss    : 1.3185
Cond loss      : 1.2832


### 9.3 Autoregressive Language Model (GPT-style)

Next-token prediction over a discrete vocabulary. `causal=True` masks future tokens so each position only attends to its past.

**Stack:** `DiscreteTokenTokenizer` embeds integer IDs -> `TokenToTokenTransformer(causal=True)` refines them -> a linear head projects to `vocab_size` logits.

**At inference:** feed the current sequence, take the last token's logit, sample the next ID, append, repeat.

In [27]:
VOCAB = 8192
EMB   = 256
MAX_L = 512

embedder  = DiscreteTokenTokenizer(vocab_size=VOCAB, embedding_dim=EMB)
backbone  = TokenToTokenTransformer(
    input_dim=EMB, output_dim=EMB, embedding_dim=EMB,
    depth=6, num_heads=8,
    causal=True,
    positional_encoding="rope",   # RoPE handles variable lengths without a fixed max
)
lm_head   = nn.Linear(EMB, VOCAB, bias=False)

# Training step
B, T  = 2, 64
ids   = torch.randint(1, VOCAB, (B, T))       # (B, T)  -  integer token IDs
emb   = embedder(ids)                          # (B, T, EMB)
feats = backbone(emb)                          # (B, T, EMB)
logits = lm_head(feats)                        # (B, T, VOCAB)

# Shift: predict token t+1 from tokens 0..t
loss = F.cross_entropy(
    logits[:, :-1].reshape(-1, VOCAB),
    ids[:, 1:].reshape(-1),
)
print(f"ids    : {ids.shape}")
print(f"logits : {logits.shape}")
print(f"LM loss: {loss.item():.4f}")

# Greedy autoregressive decode (one step)
context = ids[:, :10]                          # seed with first 10 tokens
with torch.no_grad():
    next_logit = lm_head(backbone(embedder(context)))[:, -1, :]   # last position
    next_id    = next_logit.argmax(dim=-1)
print(f"Next predicted token IDs: {next_id}")

ids    : torch.Size([2, 64])
logits : torch.Size([2, 64, 8192])
LM loss: 9.0020
Next predicted token IDs: tensor([2776, 2281])


### 9.4 Time-Series / Audio Classification

Classify a variable-length sensor or audio sequence into one of N classes.

**Stack:** `TokenToClassTransformer`  -  continuous input features -> transformer encoder -> mean-pool -> MLP -> logits.

Typical inputs: IMU (6-axis), STFT magnitude, mel spectrogram frames, ECG samples.

In [28]:
N_CLASSES  = 8
INPUT_DIM  = 80   # e.g. 80-bin mel spectrogram

classifier = TokenToClassTransformer(
    input_dim=INPUT_DIM, num_classes=N_CLASSES,
    embedding_dim=128, depth=4, num_heads=4,
    positional_encoding="sinusoidal",
    pooling="mean",       # average valid frames -> class logits
)

# Variable-length sequences in the same batch (padded)
B = 4
# Lengths: 120, 95, 150, 80 frames  -  simulate with a mask
T_max = 150
x    = torch.randn(B, T_max, INPUT_DIM)
mask = torch.zeros(B, T_max, dtype=torch.bool)
for i, length in enumerate([120, 95, 150, 80]):
    mask[i, :length] = True

logits = classifier(x, mask=mask)         # (B, N_CLASSES)
labels = torch.randint(0, N_CLASSES, (B,))
loss   = F.cross_entropy(logits, labels)

print(f"Input  : {x.shape}  (padded to T_max={T_max})")
print(f"Logits : {logits.shape}")
print(f"CE loss: {loss.item():.4f}")

Input  : torch.Size([4, 150, 80])  (padded to T_max=150)
Logits : torch.Size([4, 8])
CE loss: 2.0673


### 9.5 Contrastive Dual Encoder (CLIP-style)

Two independent encoders map sequences from different modalities into a shared embedding space, trained with InfoNCE loss so matching pairs are close and non-matching pairs are far apart.

**Stack:** two `TokenToVectorTransformer` instances (one per modality) -> L2-normalised embeddings -> cosine similarity matrix -> contrastive loss.

Works for audio-text, sensor-label, image-caption, or any two-modality pairing.

In [29]:
EMBED_DIM = 128

# Modality A: e.g. 80-dim mel spectrogram frames
encoder_a = TokenToVectorTransformer(
    input_dim=80, output_dim=EMBED_DIM, embedding_dim=256,
    depth=4, num_heads=4, positional_encoding="sinusoidal", pooling="mean",
)
# Modality B: e.g. 16-dim continuous label sequence or text features
encoder_b = TokenToVectorTransformer(
    input_dim=16, output_dim=EMBED_DIM, embedding_dim=128,
    depth=4, num_heads=4, positional_encoding="sinusoidal", pooling="mean",
)

temperature = nn.Parameter(torch.tensor(0.07))   # learnable temperature

B = 8
seq_a = torch.randn(B, 50, 80)   # modality A sequences
seq_b = torch.randn(B, 20, 16)   # paired modality B sequences

emb_a = F.normalize(encoder_a(seq_a), dim=-1)   # (B, EMBED_DIM)
emb_b = F.normalize(encoder_b(seq_b), dim=-1)   # (B, EMBED_DIM)

# Cosine similarity matrix scaled by temperature
sim = (emb_a @ emb_b.T) / temperature.exp()     # (B, B)

# Symmetric InfoNCE: diagonal = positive pairs
targets = torch.arange(B)
loss = (F.cross_entropy(sim, targets) + F.cross_entropy(sim.T, targets)) / 2

print(f"Embedding A : {emb_a.shape}  norm={emb_a.norm(dim=-1).mean():.3f}")
print(f"Embedding B : {emb_b.shape}  norm={emb_b.norm(dim=-1).mean():.3f}")
print(f"Sim matrix  : {sim.shape}")
print(f"InfoNCE loss: {loss.item():.4f}")

Embedding A : torch.Size([8, 128])  norm=1.000
Embedding B : torch.Size([8, 128])  norm=1.000
Sim matrix  : torch.Size([8, 8])
InfoNCE loss: 2.0800


### 9.6 Object Detection (DETR-style)

An image encoder produces a token sequence; a fixed set of learned query slots decode that into a fixed number of bounding box predictions (one box per query slot). Non-matched slots predict a "no-object" class.

**Stack:** `PatchTransformerND(output_mode="tokens")` encodes patches -> `QuerySetDecoder` decodes N slots -> each slot outputs `(x, y, w, h, score, class)`.

The DETR training uses Hungarian matching between predicted and ground-truth boxes  -  the forward pass shown here produces the raw predictions.

In [30]:
N_QUERIES  = 100   # max detectable objects per image
N_CLASSES  = 20   # COCO-style: 20 object classes + 1 background
BOX_DIM    = 4    # (cx, cy, w, h) normalised to [0, 1]
OUT_DIM    = BOX_DIM + N_CLASSES + 1   # box coords + class logits (incl. no-object)

# Image backbone: 2D ViT producing patch tokens
image_encoder = PatchTransformerND(
    input_dim=2, in_channels=3, out_channels=256, patch_size=16,
    embedding_dim=256, depth=6, num_heads=8,
    output_mode="tokens",           # returns (B, n_patches, 256)
    positional_encoding="sinusoidal",
)

# Pool patch tokens to a single latent for the query decoder
# (In original DETR the decoder cross-attends to all patches;
#  here we use a simpler mean-pool -> QuerySetDecoder for clarity.)
pooler  = TokenPooling(mode="mean")
decoder = QuerySetDecoder(
    latent_dim=256, output_dim=OUT_DIM,
    num_queries=N_QUERIES, query_dim=128,
)

B = 2
images = torch.randn(B, 3, 224, 224)

patch_tokens = image_encoder(images)           # (B, 196, 256)
latent       = pooler(patch_tokens)            # (B, 256)
predictions  = decoder(latent)                 # (B, N_QUERIES, OUT_DIM)

boxes  = predictions[..., :BOX_DIM].sigmoid() # normalise coords to (0, 1)
cls_logits = predictions[..., BOX_DIM:]       # (B, N_QUERIES, N_CLASSES+1)

print(f"Image        : {images.shape}")
print(f"Patch tokens : {patch_tokens.shape}")
print(f"Latent       : {latent.shape}")
print(f"Predictions  : {predictions.shape}")
print(f"  Boxes      : {boxes.shape}       (cx, cy, w, h) in [0,1]")
print(f"  Classes    : {cls_logits.shape}  (class logits incl. no-object)")

Image        : torch.Size([2, 3, 224, 224])
Patch tokens : torch.Size([2, 196, 256])
Latent       : torch.Size([2, 256])
Predictions  : torch.Size([2, 100, 25])
  Boxes      : torch.Size([2, 100, 4])       (cx, cy, w, h) in [0,1]
  Classes    : torch.Size([2, 100, 21])  (class logits incl. no-object)
